# Formula E — video verifier check (video-direct)

Validates the shipping approach: on a telemetry stop, point Gemini straight at each mosaic in the bucket and pass `videoMetadata` start/end offsets so it decodes only that window — **no download, no frame extraction, no warm-up.** It sweeps all six camera groups and judges the track state (persistent **blockage** vs **cleared**).

**Run-all** sets up the Gemini client and checks the three known cases — expect **blocked / cleared / blocked** for 693 / 1373 / 1780. Needs this project's ADC with read on `MOSAICS_BASE`.

## 1 · Config
Project ID auto-detects from the environment; the mosaics path derives from it.

In [ ]:
# Project auto-detects from where you're running (env → ADC → gcloud). Override by
# setting PROJECT_ID = "your-project" below.
PROJECT_ID = ""
if not PROJECT_ID:
    import os
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "") or ""
if not PROJECT_ID:
    try:
        import google.auth
        PROJECT_ID = google.auth.default()[1] or ""
    except Exception:
        pass
if not PROJECT_ID:
    try:
        import subprocess
        PROJECT_ID = subprocess.run(["gcloud","config","get-value","project"],
                                    capture_output=True, text=True).stdout.strip()
    except Exception:
        pass
assert PROJECT_ID, "Could not auto-detect PROJECT_ID — set it manually above."

# Your project's staged mosaics (same-region reads are free). Swap to the shared
# class bucket if needed:  MOSAICS_BASE = "gs://class-demo/formula-e/r10/mosaics"
MOSAICS_BASE = f"gs://{PROJECT_ID}-fe-mosaics/mosaics"
GEM_MODEL = "gemini-3.5-flash"
print("project:", PROJECT_ID, "| mosaics:", MOSAICS_BASE)

## 2 · Gemini client (Vertex + ADC)

In [ ]:
import json, re, time
try:
    from google import genai
    from google.genai import types
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'google-genai'], check=True)
    from google import genai
    from google.genai import types
import google.auth

_proj = PROJECT_ID or google.auth.default()[1]
GEM = genai.Client(vertexai=True, project=_proj, location='global')
print('Gemini ready via Vertex project', _proj, 'model', GEM_MODEL)

def _parse(text):
    s = (text or '').strip()
    a, b = s.find('{'), s.rfind('}')
    if a != -1 and b > a:
        try: return json.loads(s[a:b+1])
        except Exception: pass
    return {}

## 3 · Camera groups
The six 2x2 mosaics, each four track-consecutive cameras (TL, TR, BL, BR).

In [ ]:
GROUPS = {
    'grp_01_cam01_cam02_cam03_cam04': ['Cam01','Cam02','Cam03','Cam04'],
    'grp_02_cam05_cam06_cam07_cam08': ['Cam05','Cam06','Cam07','Cam08'],
    'grp_03_cam09_cam10_cam11_cam12': ['Cam09','Cam10','Cam11','Cam12'],
    'grp_04_cam13_cam14_cam15_cam16': ['Cam13','Cam14','Cam15','Cam16'],
    'grp_05_cam17_cam18_cam19_cam20': ['Cam17','Cam18','Cam19','Cam20'],
    'grp_06_cam21_cam22_cam23_cam24': ['Cam21','Cam22','Cam23','Cam24'],
}
print(len(GROUPS), 'groups')

# 2024 liveries (demo cars) — a hint so the verifier can cross-check identity
LIVERIES = {7:'Maserati - dark blue with an orange rear flash',
            9:'Jaguar - black and white', 17:'Andretti - white, red and blue',
            23:'Nissan - red, white and black', 48:'Mahindra - matt red and silver'}

## 4 · The verifier — hand Gemini a gs:// slice
For each group: a `gs://` video Part with start/end offsets (the window around the stop) + a grounded persistence prompt. Judges **track state**, not car identity; sweeps all groups and takes the strongest blockage.

In [ ]:
def verify_video_direct(t, cars=None, lead=10, tail=50):
    """Persistence sweep over gs:// video slices (no download). `cars` adds a livery hint."""
    start, end = max(0, t - lead), t + tail
    hint = ''
    if cars:
        who = ', '.join(f"#{c} ({LIVERIES.get(c,'livery unknown')})" for c in cars)
        hint = (f'Telemetry says the car(s) likely involved are: {who}. If you see a stopped car, use its '
                'LIVERY/colour AND its car NUMBER (if clearly readable) to say whether it matches. If you '
                'cannot clearly read a number, do NOT guess one - just describe the colour/livery.\n')
    print(f"\n### @ t={t}s  (gs:// slice {start}s-{end}s, all {len(GROUPS)} groups)")
    verdicts = []
    for gid, cams in GROUPS.items():
        uri = f"{MOSAICS_BASE}/{gid}.mp4"
        prompt = (
            'You are a race-control video verifier deciding whether a SAFETY CAR is warranted.\n'
            f'Telemetry flagged a car possibly stopped near here around race time ~{t}s.\n'
            + hint +
            f'This is a ~{end-start}s CCTV clip - a 2x2 mosaic: TL={cams[0]}, TR={cams[1]}, BL={cams[2]}, BR={cams[3]}.\n'
            'Judge the TRACK STATE by the END (the call is about the track, not which car):\n'
            '- Car STILL stopped/stranded on/beside the line at the end: blockage=true, cleared=false.\n'
            '- Car appeared but DROVE AWAY / recovered / line clear by the end: blockage=false, cleared=true.\n'
            '- No stopped car at any point: blockage=false, cleared=false.\n'
            'Note if other cars are moving.\n'
            'JSON: {"blockage": bool, "cleared": bool, "panel": "TL|TR|BL|BR|none", "feed_live": bool, '
            '"seen_car": <car number if clearly readable else null>, "what_you_see": str, "confidence": number}')
        try:
            vpart = types.Part(file_data=types.FileData(file_uri=uri, mime_type='video/mp4'),
                               video_metadata=types.VideoMetadata(start_offset=f'{start}s', end_offset=f'{end}s'))
            resp = GEM.models.generate_content(model=GEM_MODEL,
                contents=[types.Content(role='user', parts=[vpart, types.Part(text=prompt)])],
                config=types.GenerateContentConfig(temperature=0.2, response_mime_type='application/json'))
            d = _parse(resp.text)
        except Exception as e:
            print(f"  {gid:34} ERROR: {e}"); continue
        state = 'BLOCKED' if d.get('blockage') else ('cleared' if d.get('cleared') else '-')
        seen = d.get('seen_car')
        print(f"  {gid:34} {state:8} panel={str(d.get('panel','?')):5} seen_car={seen} conf={d.get('confidence','?')}")
        if d.get('blockage') or d.get('cleared'):
            print(f"        -> {str(d.get('what_you_see',''))[:150]}")
        verdicts.append((gid, d))
    blocked = [g for g, d in verdicts if d.get('blockage')]
    print('  VERDICT: ' + ('BLOCKAGE in ' + ', '.join(blocked) if blocked else 'no blockage (cleared)'))
    return verdicts

## 5 · Validate — three known cases
Expect **blocked / cleared / blocked**. (Ground truth: #7 Günther and #23 Fenestraz retired; Evans #9 had a big decel at 1373s but finished 6th.)

In [ ]:
_ = verify_video_direct(693, cars=[7])     # Gunther  -> BLOCKED, should ID #7 (blue Maserati)
_ = verify_video_direct(1373, cars=[9])    # Evans    -> CLEARED
_ = verify_video_direct(1780, cars=[48])   # Mortara  -> BLOCKED

## 6 · Timing
Real per-verify latency (six groups, one stop) — no download/extract, so this is what the live agent pays.

In [ ]:
t0 = time.time(); _ = verify_video_direct(693)
print(f"\nelapsed: {time.time()-t0:.1f}s for 6 groups")

## 7 · Time-alignment check
Proves an offset of `Ns` really lands on race-second `N`: seek there, read the burned-in clock, and compare to green flag + offset. Green flag = 13:04:00 UTC = **15:04:00 Berlin (CEST)**.

In [ ]:
from datetime import datetime, timedelta
GREEN_LOCAL = datetime(2024, 5, 12, 15, 4, 0)   # Berlin local time at race-second 0

def check_alignment(t, lead=10):
    start = max(0, t - lead)
    gid = next(iter(GROUPS)); uri = f'{MOSAICS_BASE}/{gid}.mp4'
    prompt = ('Each CCTV panel has a clock (HH:MM:SS) burned into it. Read the clock shown in the FIRST frame of this clip and return JSON: {"burned_in_time": "HH:MM:SS"}')
    try:
        vpart = types.Part(file_data=types.FileData(file_uri=uri, mime_type='video/mp4'),
                           video_metadata=types.VideoMetadata(start_offset=f'{start}s', end_offset=f'{start+3}s'))
        resp = GEM.models.generate_content(model=GEM_MODEL,
            contents=[types.Content(role='user', parts=[vpart, types.Part(text=prompt)])],
            config=types.GenerateContentConfig(temperature=0, response_mime_type='application/json'))
        seen = str(_parse(resp.text).get('burned_in_time', '?'))
    except Exception as e:
        print(f'  offset {start}s -> ERROR: {e}'); return
    expected = (GREEN_LOCAL + timedelta(seconds=start)).strftime('%H:%M:%S')
    ok = seen[:5] == expected[:5]   # match to the minute (seek lands within a few seconds)
    print(f"  offset {start:>4}s  ->  Gemini reads {seen:>8}   expected ~{expected}   {'ALIGNED' if ok else 'CHECK <-'}")

for t in (693, 1373, 1780):
    check_alignment(t)